# Machine Learning Baseline - Crime Rate Prediction

Objetivo deste notebook:

- consumir a base analitica oficial do PostgreSQL/Data Warehouse;
- preparar features socioeconomicas e educacionais;
- treinar um primeiro modelo de regressao linear;
- avaliar o erro do modelo;
- futuramente salvar previsoes/resultados para uso no BI.

Importante: a limpeza, padronizacao e integracao dos dados acontecem no PostgreSQL, principalmente nos schemas `raw` e `dw`. Este notebook nao deve substituir o pipeline SQL do DW.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd

from sqlalchemy import create_engine
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

## 2. Database Connection

Dentro do Docker Compose, o host do PostgreSQL e `postgres-service`.

Se estiver executando fora do container, use `localhost` no lugar de `postgres-service`.

In [ ]:
DB_USER = 'postgres'
DB_PASSWORD = 'postgres'
DB_HOST = 'postgres-service'
DB_PORT = '5432'
DB_NAME = 'seguranca_publica'

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

## 3. Load Analytical Dataset From DW

A granularidade esperada e:

`1 linha = 1 municipio/capital + 1 ano`

In [ ]:
query = """
SELECT
    fato.ano,
    municipio.codigo_municipio,
    municipio.nome_municipio,
    uf.sigla_uf,
    regiao.nome_regiao,
    fato.populacao_total,
    fato.populacao_crescimento_pct,
    fato.idhm,
    fato.idhm_renda,
    fato.idhm_educacao,
    fato.idhm_longevidade,
    fato.ideb,
    fato.fluxo,
    fato.aprendizado,
    fato.nota_mt,
    fato.nota_lp,
    fato.taxa_crimes_100k,
    fato.risco_indice
FROM dw.fato_municipio_ano AS fato
JOIN dw.dim_municipio AS municipio
    ON municipio.id_municipio_dw = fato.id_municipio_dw
JOIN dw.dim_uf AS uf
    ON uf.id_uf_dw = municipio.id_uf_dw
JOIN dw.dim_regiao AS regiao
    ON regiao.id_regiao_dw = uf.id_regiao_dw
WHERE fato.taxa_crimes_100k IS NOT NULL;
"""

df = pd.read_sql(query, engine)
df.head()

## 4. Data Quality Check For Modeling

In [ ]:
df.info()
df.isna().sum().sort_values(ascending=False)

## 5. Feature Selection

Primeiro baseline simples:

- variaveis socioeconomicas;
- variaveis educacionais;
- populacao e crescimento populacional;
- ano como variavel temporal simples.

Target inicial: `taxa_crimes_100k`.

In [ ]:
target = 'taxa_crimes_100k'

features = [
    'ano',
    'populacao_total',
    'populacao_crescimento_pct',
    'idhm',
    'idhm_renda',
    'idhm_educacao',
    'idhm_longevidade',
    'ideb',
    'fluxo',
    'aprendizado',
    'nota_mt',
    'nota_lp',
]

df_modelo = df[features + [target]].dropna().copy()

X = df_modelo[features]
y = df_modelo[target]

df_modelo.shape

## 6. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
)

X_train.shape, X_test.shape

## 7. Train Linear Regression

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)

## 8. Evaluation

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metricas = pd.DataFrame({
    'metrica': ['MAE', 'RMSE', 'R2'],
    'valor': [mae, rmse, r2],
})

metricas

## 9. Coefficients Interpretation

In [ ]:
coeficientes = pd.DataFrame({
    'feature': features,
    'coeficiente': modelo.coef_,
}).sort_values('coeficiente', ascending=False)

coeficientes

## 10. Next Steps

- validar se o split temporal faz mais sentido que o split aleatorio;
- testar regularizacao com Ridge/Lasso;
- adicionar features de defasagem por municipio;
- salvar previsoes em uma tabela propria para BI;
- comparar modelos de baseline com modelos mais robustos.